# ICE data


In [ ]:
# imports
import polars as pl  # polars is much faster than pandas for large data
import numpy as np

from helpers import get_latest

In [ ]:
# TODO: what stories were made from these? other than the criminal record one with WyoFile
# TODO: get style guide
# TODO: did we use the processed or original data? - looks like original, but can we use processed to aid in duplicate detection? - ask Fish
# TODO: I think we should start open-sourcing our methodology/scripts
# TODO: are we working with Wyoming again? - no
# TODO: do we want a repeat of last year's graphics, or new stuff? longer term data is available, do we want to make use of it? eg, instead of bar graph comparing 2 years, have faceted line graph for each criminality or other breakdown showing digression from stable pattern
# TODO: what kind of comments and feedback did we get on last year's story?
# TODO: which field office made the arrest is the most reliable column - Denver Field Office covers both Wyoming and Colorado
# TODO: Berkeley deportation data project is very responsive - zero in on Colorado with the caveats
# TODO: criminal history - conviction, charge, other (no charge or conviction) - if you want to know most serious charge (MSC) - must merge with detentions - ask Fish for this
# TODO: timeline: within a week, about the number of arrests and most serious convictions

## Source

[Deportation Data Project](https://deportationdata.org/data/ice.html)

## Past stories

[ICE arrests](https://coloradosun.com/2025/12/31/ice-arrests-2025-data-deportation-data-project/)


# Arrests

This is the main table we are working with to start. From the description: "Records every time ICE arrests someone, whether or not that arrest results in a decision to detain the person; or issues a Notice to Appear (NTA), the document that starts a deportation case. Note that other agencies also issue NTAs, and those would not appear as arrests in this table. We treat "apprehensions," "arrests," and "administrative arrests" as synonyms."


In [ ]:
arrests = pl.read_excel(get_latest("data", "arrests"))
sorted(list(arrests.schema.keys()))

Appears to be a data issue with impossible date of arrest


In [ ]:
arrests = arrests.with_columns(
    arrest_year=pl.col("apprehension_date").dt.year(),
    arrest_month=pl.col("apprehension_date").dt.month(),
    arrest_day=pl.col("apprehension_date").dt.day(),
)

arrests["apprehension_date"].sort().value_counts()

## Cleaning

First filter for blanks in the Apprehension State field. See if Wyoming or Colorado show up in the Facility State, and then if Detention Facility or Apprehension Landmark Site appear to be in Wyoming or Colorado, and update the Apprehension State accordingly.

Note: facility is probably unrelated to apprehension - people have been being shipped to random facilities


In [ ]:
prob_co_arrests = arrests["apprehension_state"].is_null() & (
    arrests["apprehension_site_landmark"].str.contains("DENVER")
)

arrests.filter(prob_co_arrests)["apprehension_site_landmark"].value_counts()

arrests = arrests.with_columns(
    apprehension_state=pl.when(prob_co_arrests)
    .then(pl.lit("COLORADO"))
    .otherwise(pl.col("apprehension_state"))
)

Then take a look at the other states listed in the Apprehension State field and remove those that don't belong. (More on this later.)

TODO: what does this mean


Then search for unique identifiers using pivot table counts and examine those that are more than one. If they occurred in different years, keep both. If they occurred within a day or two get rid of the older one (often there's removal on one of them, keep that one).

If you plan to compare 2025 to 2025 numbers, take out only the data for the dates available in 2025 (Jan. 20 to the most recent month and date). Don't forget to take out the records from Jan.1-19, 2025. (Why?)

Then it's time for analysis. (will update with a new notebook at some point.)


Then remove duplicates, keep latest or removal or same day most info UNLESS different years then keep both

TODO: this used to be based on ID, but isn't it meaningful that someone is arrested more than once?

Changed to use the "duplicate_likely" column provided by Deportation Data Project


In [ ]:
arrests_duplicates = (
    arrests.filter(arrests["duplicate_likely"])
    .sort(["unique_identifier", "apprehension_date_time"])
    .unique(subset=["unique_identifier"], keep="last")
)

arrests = arrests.filter(~arrests["duplicate_likely"]).vstack(arrests_duplicates)

# Detentions

This is the detentions section. The description: "Records each detention from book-in to book-out at a given detention center for a given individual; most individuals have more than one row in the table because they are transferred between detention centers. Individuals may also be detained more than once and therefore have more than one "stay" in detention. We explain further in the relevant field definitions."
This spreadsheet has two tables that must be combined.


In [ ]:
# create a list of prior two tables, then concantenate them to one; check the record totals
detentions_sheets = pl.read_excel(
    get_latest("data", "detention-stays"),
    sheet_id=0,
    schema_overrides={"initial_bond_set_date": pl.Date},
)
detentions = pl.concat([v for k, v in detentions_sheets.items()])
print(detentions.shape)
detentions.schema

In [ ]:
# this gets a list of the detention facilities
det_facility = (
    detentions.groupby(["Detention Facility"]).size().reset_index(name="counts")
)
det_facility.write_csv("det_facility.csv")

In [ ]:
# these are the colorado, wyoming facilities thus far
# TODO: where do we get this?

wy_facilities = [
    "CASPER HOLDROOM",
    "CHEYENNE HOLDROOM",
    "LARAMIE COUNTY JAIL",
    "NATRONA COUNTY JAIL",
    "SWEETWATER COUNTY JAIL",
]

co_facilities = [
    "ALAMOSA HOLDROOM",
    "ARAPAHOE COUNTY JAIL",
    "AURORA CITY JAIL",
    "COLO DEPT OF CORRECTIONS",
    "COLO SPRINGS DEN HSI HOLD",
    "CRAIG HOLDROOM",
    "DENVER CONTRACT DETENTION FACILITY",
    "DENVER COUNTY JAIL",
    "DENVER HEALTH MEDICAL CENTER",
    "DENVER HOLD ROOM",
    "DOUGLAS COUNTY DETENTION CENTER",
    "DURANGO HOLDROOM",
    "GLENWOOD SPRINGS HOLDROOM",
    "GRAND JUNCTION HOLDROOM",
    "JACKSON COUNTY SHERIFF",
    "JEFFERSON COUNTY JAIL",
    "MESA COUNTY JAIL",
    "MOFFAT COUNTY JAIL",
    "OTERO COUNTY DETENTION",
    "PUEBLO COUNTY JAIL",
    "PUEBLO HOLDROOM",
    "SUMMIT COUNTY JAIL",
    "TELLER COUNTY JAIL",
    "UCHEALTH UNV HOSP OF COUINTA COUNTY JAIL",
]

In [ ]:
# this uses the list of colo, wyo to extract the info from the larger detention facility list
co_detain = detentions.filter(pl.col("detention_facility_first").is_in(co_facilities))
co_detain.write_csv("co_wyo_detain.csv")
co_detain.head()

## Removals

From the description: "Records every deportation that ICE conducts, with a row for each individual deportation. An individual only has more than one row if that individual was deported more than once. Note that expulsions may occur directly at the border, by CBP, without involving ICE."


In [ ]:
removals = pl.read_excel(get_latest("data", "removals"))
removals.head()

In [ ]:
co__wyo_remove = removals[
    (removals["Docket AOR"] == "Denver Area of Responsibility")
    | (removals["Apprehension State"] == "Colorado")
    | (removals["Apprehension State"] == "Wyoming")
]

In [ ]:
co__wyo_remove.head()

In [ ]:
co__wyo_remove.write_csv("co_wyo_remove.csv")

# Detainers

From the description: "Records all requests to state, county, and municipal jails and prisons either for a person to be held on a detainer or for a notification of release date and time. A detainer is a request to a local jail to hold someone for 48 hours beyond when they otherwise would be released so that ICE can make an arrest in the jail while the individual remains detained."


In [ ]:
detainer = pl.read_excel(get_latest("data", "detainer"))
detainer.schema

In [ ]:
felon_lookup = detainer[
    [
        "Unique Identifier",
        "MSC Conviction Date",
        "Detention Facility",
        "Facility State",
        "Prior Felony Yes No",
        "Violent Misdemeanor Yes No",
        "Aggravated Felony Yes No",
    ]
].drop_duplicates(subset="Unique Identifier")
felon_lookup.head()

In [ ]:
# this adds more information about conviction date, facility and more; mostly used the conviction date
arrests_supplement = arrests_with_charge.merge(
    felon_lookup, on="Unique Identifier", how="left"
)
arrests_supplement.head()


In [ ]:
arrests_supplement.write_csv("arrests_plus.csv")

In [ ]:
detainer.groupby(["Facility AOR"]).size().reset_index(name="counts")

In [ ]:
detainer.groupby(["Facility State"]).size().reset_index(name="counts")

In [ ]:
co_wyo_detainer = detainer[
    (detainer["Facility AOR"] == "Denver Area of Responsibility")
    | (detainer["Facility State"] == "Colorado")
    | (detainer["Facility State"] == "Wyoming")
]
co_wyo_detainer.head()

In [ ]:
co_wyo_detainer.write_csv("co_wyo_detainer.csv")

# Encounters

From the description: "Records every time ICE Enforcement and Removal Operations encounters a person, i.e. considers whether to take enforcement action against a person. This need not mean a physical encounter. Most notably, every time ICE processes a match between FBI book-in information (i.e. to a jail or prison) and ICE database information, that match is logged as an ICE encounter. Generally, if an individual appears in the detainers or arrests table, that individual should appear in this table. An individual might appear in the removals or detentions tables without appearing in the encounters data if Customs and Border Protection initially encounters the person. This is both the largest and the sparsest of the tables, and in many cases, encounters lack a unique ID because the individual lacked an A number (A numbers are generally only given to people with immigrant visas or when they are processed for deportation proceedings)."


In [ ]:
encounters = pl.read_excel(get_latest("data", "encounters"))
encounters.schema

In [ ]:
encounters.groupby(["Responsible AOR"]).size().reset_index(name="counts")

In [ ]:
co_encounter = encounters[
    encounters["Responsible AOR"] == "Denver Area of Responsibility"
]
co_encounter.head()

In [ ]:
co_encounter.write_csv("co_encounter.csv")

In [ ]:
# have not looked at this closely
encounters.group_by(["Responsible Site"]).size().reset_index(name="counts")

In [ ]:
# TODO: transfer the excel cleaning steps to python
# TODO: what is "wrong date"?


## Historical data


In [ ]:
hd1_files = [f for f in os.listdir("data") if "ERO Apprehension" in f]
hd1 = pl.DataFrame()
for f in tqdm(hd1_files):
    df = pl.read_excel(
        os.path.join("data", f),
        read_options={"header_row": 4},
        schema_overrides={
            "Fugitive as of Date": pl.Datetime,
            "Departed Date": pl.Datetime,
        },
    )
    # print(df.head())
    hd1 = hd1.vstack(df)
print(hd1.shape)
print(sorted(list(hd1.schema.keys())))
hd1.head()

We can concatenate the datasets because the end of the historical dataset exactly coincides with the start of the next.

In [ ]:
print(hd1["Apprehension Date And Time"].min(), hd1["Apprehension Date And Time"].max())

## Cleaning

Remove duplicates if same year

TODO: look at their code for detecting duplicates

In [ ]:
hd1 = hd1.with_columns(
    pl.col("Apprehension Date And Time").dt.date().alias("apprehension_date"),
    pl.col("Apprehension Date And Time").dt.year().alias("arrest_year"),
    pl.col("Apprehension Date And Time").dt.month().alias("arrest_month"),
    pl.col("Apprehension Date And Time").dt.day().alias("arrest_day"),
)

In [ ]:
hd1 = hd1.filter(hd1["Sequence Number/Unique Identifier"].is_not_null()).unique(
    subset=["Sequence Number/Unique Identifier", "arrest_year"], keep="last"
)

# Research Questions


In [ ]:
# TODO: follow up with fish on analysis notebook
# gender, age
# citizenship, third country deportations
# per month
# criminality
# TODO: fish comment about which days there weren't any arrests, pulling out holidays
# TODO: Aurora Detention Center operated by Geogroup - for the Aurora specific story - Taylor idea about https://www.nytimes.com/interactive/2025/12/22/us/trump-immigration-deportation-network-ice-arrests.html - inspiration https://coloradotimesrecorder.com/2026/03/after-co-times-recorder-revealed-secret-detention-centers-in-co-27-lawmakers-call-on-ice-for-immediate-transparency/77113/ - Denver Health


# Aurora story
# detentions stays/stints
# TODO: ask DDP if the historical data is comparable for this use case - ask Fish if the historical data changes from release to release


### Number of arrests

In [ ]:
# This gets arrests with Denver area of responsibility and/or the state of Colorado/Wyoming
co_arrests = arrests.filter(
    (arrests["apprehension_aor"] == "Denver Area of Responsibility")
    | (arrests["apprehension_state"] == "COLORADO")
    # | (arrests['apprehension_state'] == 'WYOMING')
)

print(co_arrests.shape)
co_arrests.head()

In [ ]:
co_hist = hd1.filter(
    (pl.col("State")== "COLORADO")
    | (pl.col("Apprehension Landmark").str.contains("DENVER"))
    | (pl.col("Apprehension AOR").str.contains("Denver Area of Responsibility"))
)
print(co_hist.shape)
co_hist.write_csv("output/co_hist.csv")
co_hist.group_by("arrest_year").agg(pl.count())

co_hist = co_hist.with_columns(
    unique_identifier=pl.col("Sequence Number/Unique Identifier").cast(pl.String),
    msc_charge=pl.col("Most Serious Criminal Conviction"),
    apprehension_criminality=pl.when(
        pl.col("Most Serious Criminal Conviction").is_not_null()
    )
    .then(pl.lit("1 Convicted Criminal"))
    .when(pl.col("Most Serious Criminal Charge").is_not_null())
    .then(pl.lit("2 Pending Criminal Charges"))
    .otherwise(pl.lit("3 Other Immigration Violator")),
)
print(co_hist["apprehension_criminality"].value_counts())
print(co_arrests["apprehension_criminality"].value_counts())

co_all = co_arrests.join(charge, on="unique_identifier", how="left")[
    "unique_identifier",
    "apprehension_date",
    'apprehension_criminality',
    "arrest_day",
    "arrest_month",
    "arrest_year",
    "msc_charge",
].vstack(
    co_hist[
        "unique_identifier",
        "apprehension_date",
        'apprehension_criminality',
        "arrest_day",
        "arrest_month",
        "arrest_year",
        "msc_charge",
    ]
)

co_all.write_csv("output/co_all.csv")

### Number of arrests

Cumulative line charts to make interpretation clearer across years, and make total as well as trends visible on the same chart.

Therefore, we will limit to the comparable months per year.

For comparing to prior year, ensure that data for prior year up to current month is kept


In [ ]:
co_latest = co_all.filter(
    # less than max month/day available in current dataset
    ((co_all["arrest_month"] < co_all["apprehension_date"].max().month)
    | (
        (co_all["arrest_month"] == co_all["apprehension_date"].max().month)
        & (co_all["arrest_day"] <= co_all["apprehension_date"].max().day)
    )) & ~(
        
        (co_all["arrest_month"] == 1)
        & (co_all["arrest_day"] < 20)
    ) # account for inauguration day
)

by_date = co_latest.with_columns(
    arrest_month_day=pl.col("arrest_month").cast(pl.Utf8).str.zfill(2)
    + "-"
    + pl.col("arrest_day").cast(pl.Utf8).str.zfill(2)
)

# each record is an arrest for a unique person
by_date = by_date.group_by(["arrest_year", "arrest_month_day"]).agg(pl.count())
by_date = by_date.sort(["arrest_year", "arrest_month_day"]).with_columns(
    cumsum=pl.col("count").fill_null(0).cum_sum().over("arrest_year")
)

by_date = by_date.pivot(
    index=["arrest_month_day"], columns="arrest_year", values="cumsum"
)

by_date.write_csv(f"output/by_date_{datetime.today().year}.csv")
by_date.sort("arrest_month_day")

In [ ]:
# get the first two years of the past administrations
by_admin = co_all.filter(
    co_all["arrest_year"].is_in([2017, 2018, 2021, 2022, 2025, 2026])
    &
    (
        # odd years are complete except for inauguration day
        (co_all["arrest_year"] % 2 == 1)
        & ~(
            (co_all["arrest_month"] == 1) & (co_all["arrest_day"] < 20)
        )  
        | (
            # even years are only complete through the max month/day available in current dataset
            (co_all["arrest_year"] % 2 == 0)
            & (co_all["arrest_month"] == co_all["apprehension_date"].max().month)
            & (co_all["arrest_day"] <= co_all["apprehension_date"].max().day)
        )
    )
).with_columns(
    admin_month=pl.col("arrest_month").cast(pl.Int32)
    + (12 * (1 - pl.col("arrest_year").cast(pl.Int32) % 2))
)

print(
    by_admin[["arrest_year", "apprehension_date", "admin_month"]]
    .sort("admin_month")
    .unique("admin_month")
)

# recode
by_admin = by_admin.with_columns(
    admin=pl.when(pl.col("arrest_year").is_in([2017, 2018]))
    .then(pl.lit("First Trump"))
    .when(pl.col("arrest_year").is_in([2021, 2022]))
    .then(pl.lit("Biden"))
    .otherwise(pl.lit("Second Trump"))
)

# each record is an arrest for a unique person
by_admin = by_admin.group_by(["admin", "admin_month"]).agg(pl.count())
by_admin = by_admin.sort(["admin", "admin_month"]).with_columns(
    cumsum=pl.col("count").fill_null(0).cum_sum().over("admin")
)

by_admin = by_admin.pivot(index=["admin_month"], columns="admin", values="cumsum")

by_admin.write_csv(f"output/by_admin_{datetime.today().year}.csv")
by_admin.sort("admin_month")

In [ ]:
totals = co_all.group_by("arrest_year").agg(pl.count())
totals.write_csv(f"output/totals_{datetime.today().year}.csv")

### Criminality

In [ ]:
crim = co_latest.with_columns(
    apprehension_criminality=pl.col("apprehension_criminality").str.replace(r"\d ", "")
)

crim = crim.group_by(["apprehension_criminality", "arrest_year"]).agg(pl.count()).pivot(
    values="count", index="arrest_year", columns="apprehension_criminality"
)

crim.write_csv(f"by_criminality_{datetime.today().year}.csv")
crim

### MSC (most serious charge)

percentage 2011-2026

- when they have conviction merge arrests with detentions on ID to get MSC
- note how many people in detention have MSC but no conviction
- FBI definition of violent/not over time
- find list of top 5 crimes since Trump's inauguration


In [ ]:
# most accurate information is most recent
charge = detentions.filter(pl.col('msc_charge').is_not_null()).sort('stay_book_out_date_time').unique(
    subset=["unique_identifier"], keep="last"
)[["unique_identifier", "msc_charge"]]

In [ ]:
fbi_violent_crime = [
# '',
 'Aggravated Assault - Family-Strongarm',
 'Aggravated Assault - Gun',
 'Aggravated Assault - Non-family-Gun',
 'Aggravated Assault - Non-family-Strongarm',
 'Aggravated Assault - Non-family-Weapon',
 'Aggravated Assault - Weapon',
#  'Amphetamine - Manufacturing',
#  'Amphetamine - Possession',
#  'Amphetamine - Sell',
#  'Assault',
#  'Battery',
#  'Burglary',
#  'Burglary - Forced Entry-Residence',
#  'Burglary - No Forced Entry-Non-Residence',
#  'Carrying Concealed Weapon',
#  'Carrying Prohibited Weapon',
#  'Cocaine',
#  'Cocaine - Possession',
#  'Cocaine - Sell',
#  'Commercial Sex',
#  'Conspiracy [use when no underlying offense, such as 18 U.S.C. SEC. 371]',
#  'Contempt Of Court',
#  'Contributing to Delinquency of Minor',
#  'Counterfeiting',
#  'Crimes Against Person',
#  'Cruelty Toward Child',
#  'Damage Property',
#  'Dangerous Drugs',
#  'Disorderly Conduct',
#  'Domestic Violence',
#  'Driving Under Influence Drugs',
#  'Driving Under Influence Liquor',
#  'Drug Possession',
#  'Drug Trafficking',
#  'Embezzle',
#  'Enticement of Minor for Prostitution',
#  'Exploitation/Enticement (Use the MIS Field to further describe offense)',
#  'Flight - Escape',
#  'Flight To Avoid (prosecution, confinement, etc.)',
#  'Forgery',
#  'Fraud',
#  'Fraud - False Statement',
#  'Fraud - Illegal Use Credit Cards',
#  'Fraud - Impersonating',
#  'Fraud By Wire',
#  'Gambling',
#  'Harassing Communication',
#  'Heroin - Sell',
#  'Heroin - Smuggle',
#  'Hit and Run',
 'Homicide',
#  'Homicide-Negligent Manslaughter-Vehicle',
#  'Identity Theft',
#  'Identity Theft (70AA)',
#  'Illegal Entry (INA SEC.101(a)(43)(O), 8USC1325 only)',
#  'Illegal Re-Entry (INA SEC.101(a)(43)(O), 8USC1326 only)',
#  'Incest With Minor',
#  'Indecent Exposure',
#  'Intimidation',
#  'Kidnapping',
#  'Larceny',
#  'Larceny - From Auto',
#  'Licensing - Registration Weapon',
#  'Liquor',
#  'Liquor - Possession',
#  'Making False Report',
#  'Marijuana - Possession',
#  'Marijuana - Sell',
#  'Marijuana - Smuggle',
#  'Narcotic Equip - Possession',
#  'Obstruct Police',
#  'Possession Forged (identify in comments)',
#  'Possession Of Weapon',
#  'Probation Violation',
#  'Property Crimes',
#  'Prostitution',
#  'Public Order Crimes',
#  'Public Peace',
#  'Racketeer Influenced and Corrupt Organizations Act (RICO)',
#  'Resisting Officer',
#  'Riot - Engaging in',
 'Robbery',
 'Robbery - Residence-Gun',
 'Robbery - Residence-Weapon',
 'Sex Assault',
 'Sex Assault - Carnal Abuse',
#  'Sex Offense',
#  'Sex Offense Against Child-Fondling',
#  'Sexual Exploitation of Minor - Material - Film',
#  'Sexual Exploitation of Minor - Material - Photograph',
#  'Sexual Exploitation of Minor - Prostitution',
#  'Sexual Exploitation of Minor - Sex Performance',
#  'Sexual Exploitation of Minor - Via Telecommunications',
#  'Shoplifting',
#  'Simple Assault',
#  'Smuggling Aliens',
#  'Statutory Rape - No Force',
#  'Stolen Property',
#  'Stolen Vehicle',
#  'Theft And Use Vehicle Other Crime',
#  'Threat Terroristic State Offenses',
#  'Traffic Offense',
#  'Trespassing',
#  'Unauthorized Use of Vehicle (includes joy riding)',
#  'Vehicle Theft',
#  'Violation of a Court Order',
#  'Voyeurism',
#  'Weapon Offense'
]

In [ ]:
# adds most serious conviction information, if present, to the arrests table
arrests_with_charge = co_latest.filter(
    pl.col("apprehension_criminality").str.contains("Convicted")
).join(charge, on="unique_identifier", how="left")[
    [
        "unique_identifier",
        "msc_charge",
        "apprehension_criminality",
        "apprehension_date",
        "arrest_year",
        "arrest_month",
        "arrest_day",
    ]
]
print(arrests_with_charge.shape)

arrests_with_charge = arrests_with_charge.with_columns(
    violent=pl.when(pl.col("msc_charge").is_in(fbi_violent_crime).fill_null(False))
    .then(pl.lit("Violent"))
    .otherwise(pl.lit("Non-violent"))
)

arrests_with_charge = arrests_with_charge.group_by(["arrest_year", "violent"]).agg(
    pl.count()
)

arrests_with_charge = arrests_with_charge.with_columns(
    pct_violent=(pl.col.count / pl.col.count.sum()).over("arrest_year") * 100,
)

arrests_with_charge = arrests_with_charge.pivot(
    index="arrest_year", columns="violent", values=["pct_violent"]
)

arrests_with_charge.write_csv("arrests_charges.csv")
arrests_with_charge.sort("arrest_year")

In [ ]:
# 17 of the people with no charge have an MSC listed
charge.join(
    co_latest.filter(pl.col("apprehension_criminality").str.contains("Other")),
    on="unique_identifier",
    how="inner",
)[["apprehension_criminality", "apprehension_date", "msc_charge"]]